# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [8]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [38]:
df_cleaned = pd.read_csv('zillow_cleaned.csv')
#test train split 
y = df_cleaned['taxvaluedollarcnt']
X = df_cleaned.drop('taxvaluedollarcnt', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [39]:
y_train.head()

49046     724130.0
72330     150785.0
56168    1046800.0
55902      53588.0
70646     251844.0
Name: taxvaluedollarcnt, dtype: float64

In [40]:
#standardize data 
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [41]:
# Add as many cells as you need
# Linear Regression 
model_linear_reg = LinearRegression()

cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear = cross_val_score(model_linear_reg, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores = -scoreLinear


print(f"Mean MAE: {mae_scores.mean():.4f}")
print(f"Standard Deviation: {mae_scores.std():.4f}")

Mean MAE: 195898.5243
Standard Deviation: 1629.0584


In [42]:
#Lasso Regression
model_lasso = Lasso(alpha=0.1) 
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_lasso = cross_val_score(model_lasso, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores_lasso = -scoreLinear_lasso


print(f"Mean MAE: {mae_scores_lasso.mean():.4f}")
print(f"Standard Deviation: {mae_scores_lasso.std():.4f}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.731e+13, tolerance: 6.923e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.478e+13, tolerance: 6.920e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

Mean MAE: 195898.4865
Standard Deviation: 1629.0580


In [43]:
#gradient boosting tree
model_gradbst = GradientBoostingRegressor()
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_grdtbst = cross_val_score(model_gradbst, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores_grdbst = -scoreLinear_grdtbst


print(f"Mean MAE: {mae_scores_grdbst.mean():.4f}")
print(f"Standard Deviation: {mae_scores_grdbst.std():.4f}")

Mean MAE: 171666.0796
Standard Deviation: 1729.7922


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

Among the three models, the Gradient Boosting Regressor produced the lowest mean MAE, indicating the best predictive performance. This result is expected, as gradient boosting is capable of capturing nonlinear relationships and interactions between features that linear models cannot. Linear Regression and Lasso Regression performed similarly, with slightly higher MAE values, suggesting that a purely linear relationship may not fully capture the complexity of the data.

The relatively small standard deviations across cross-validation folds indicate that model performance is consistent and stable. Based on these baseline results, the Gradient Boosting model appears to be the strongest starting point for further improvement. In the next steps, feature engineering and model tuning will be explored to improve performance across all models.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [45]:
# Add as many cells as you need
# Bedrooms*Bathrooms
X_train['bed_bath'] = X_train['bedroomcnt'] * X_train['bathroomcnt']

# Finished square feet / Lot size
X_train['sqft_to_lot_ratio'] = X_train['calculatedfinishedsquarefeet'] / X_train['lotsizesquarefeet']

# Taking log of 'lotsizesuarefeet' and 'calculatedfinishedsquarefeet' to account for skewness
X_train['log_lotsize'] = np.log1p(X_train['lotsizesquarefeet']) #log1 handles zeros
X_train

,Unnamed: 0,bedroomcnt,bathroomcnt,roomcnt,numberofstories,yearbuilt,calculatedfinishedsquarefeet,lotsizesquarefeet,garagecarcnt,latitude,longitude,regionidzip,poolcnt,prop_age,bed_bath,sqft_to_lot_ratio,log_lotsize,log_sqft
49046,50075,3.0,3.0,0.0,1.0,1966.0,2614.0,24741.0,0.0,34149214.0,-118622654.0,96387.0,1.0,51.0,9.0,0.105655,10.116257,7.869019
72330,73849,2.0,1.0,0.0,1.0,1950.0,610.0,9655.0,0.0,34057699.0,-118052895.0,96480.0,0.0,67.0,2.0,0.063180,9.175335,6.415097
56168,57363,5.0,4.0,9.0,1.0,1951.0,3886.0,14352.0,2.0,33770272.0,-117880409.0,97006.0,1.0,66.0,20.0,0.270764,9.571714,8.265393
55902,57092,2.0,2.0,0.0,1.0,1950.0,1412.0,4934.0,0.0,33893654.0,-118084509.0,96193.0,0.0,67.0,4.0,0.286178,8.504108,7.253470
70646,72117,3.0,2.0,6.0,1.0,1963.0,1527.0,6060.0,2.0,33800557.0,-118040735.0,96218.0,1.0,54.0,6.0,0.251980,8.709630,7.331715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,37992,3.0,2.0,6.0,1.0,1989.0,1486.0,7205.0,2.0,33492099.0,-117649457.0,96985.0,0.0,28.0,6.0,0.206246,8.882669,7.304516
6265,6425,1.0,1.0,3.0,1.0,1974.0,514.0,2178.0,1.0,34190400.0,-118886000.0,96383.0,0.0,43.0,1.0,0.235996,7.686621,6.244167
54886,56051,2.0,2.0,5.0,3.0,1979.0,1609.0,7205.0,1.0,33624500.0,-117934000.0,96981.0,1.0,38.0,4.0,0.223317,8.882669,7.383989
860,880,2.0,2.0,0.0,1.0,1980.0,1706.0,32839.0,0.0,34066000.0,-118431000.0,96005.0,1.0,37.0,4.0,0.051950,10.399403,7.442493


In [46]:
# re scale 
#standardize data 
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)

In [47]:
model_linear_reg = LinearRegression()

cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear = cross_val_score(model_linear_reg, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores = -scoreLinear


print(f"Mean MAE: {mae_scores.mean():.4f}")
print(f"Standard Deviation: {mae_scores.std():.4f}")

Mean MAE: 195261.9440
Standard Deviation: 1607.5757


In [48]:
#Lasso Regression
model_lasso = Lasso(alpha=0.1) 
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_lasso = cross_val_score(model_lasso, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores_lasso = -scoreLinear_lasso


print(f"Mean MAE: {mae_scores_lasso.mean():.4f}")
print(f"Standard Deviation: {mae_scores_lasso.std():.4f}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.705e+13, tolerance: 6.923e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.453e+13, tolerance: 6.920e+11
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

Mean MAE: 195261.9208
Standard Deviation: 1607.5747


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.526e+13, tolerance: 6.948e+11
  model = cd_fast.enet_coordinate_descent(


In [49]:
#gradient boosting tree
model_gradbst = GradientBoostingRegressor()
cv_strat = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

scoreLinear_grdtbst = cross_val_score(model_gradbst, x_train_scaled, y_train, cv=cv_strat, scoring='neg_mean_absolute_error')
mae_scores_grdbst = -scoreLinear_grdtbst


print(f"Mean MAE: {mae_scores_grdbst.mean():.4f}")
print(f"Standard Deviation: {mae_scores_grdbst.std():.4f}")

Mean MAE: 171643.2038
Standard Deviation: 1694.6999


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




Three new features were created to improve model performance: bed_bath, sqft_to_lot_ratio, and log_lotsize. These features were designed to capture interactions and reduce skew in the data that may not be fully represented in the original variables.

After adding these engineered features, the models were re-evaluated using repeated cross-validation and Mean Absolute Error (MAE). The results showed a slight improvement in model performance, particularly for Linear Regression, while Gradient Boosting showed minimal change. This suggests that while the new features added some useful information, the overall predictive power of the dataset was already strong.

The small improvement is not unexpected, as Gradient Boosting models can already capture nonlinear relationships and feature interactions automatically. However, the results demonstrate that feature engineering can still provide incremental improvements and remains an important step in the modeling process.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [5]:
# Add as many cells as you need


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [6]:
# Add as many cells as you need


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [7]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here